# wwd_i — Phase 2: train the backbone on Colab (T4)

Trains the from-scratch **BC-ResNet** backbone with **cosine-prototypical** metric
learning, runs the **few-shot probe gate** on held-out (unseen) words, and exports
a frozen `backbone.onnx`.

- **Part A — MSWC scale-up** (the real backbone): hundreds/thousands of words
  streamed from the Hugging Face hub.
- **Part B — Speech Commands bootstrap** (optional, ~24 words): the pipeline
  smoke-test; kept for reference.

**Before running:** *Runtime → Change runtime type → **T4 GPU***. This stays at
project parity — **Python 3.14 + torch 2.12** via `uv` — but installs the **CUDA**
torch build for the T4 (local dev pins CPU).


### 1. Confirm the T4 GPU


In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

### 2. Get the code

`git clone`s the repo — push `wwd_i` to GitHub first (private is fine; for a
private repo use `https://<TOKEN>@github.com/<you>/wwd_i.git`). No remote? Use the
**upload-a-zip** cell below instead.


In [ ]:
import os

REPO_URL = "https://github.com/<you>/wwd_i.git"  # <-- set me
BRANCH = "master"
os.environ["REPO_URL"] = REPO_URL
os.environ["BRANCH"] = BRANCH
!rm -rf /content/wwd_i && git clone --branch "$BRANCH" --depth 1 "$REPO_URL" /content/wwd_i
%cd /content/wwd_i
!git log --oneline -1

### 2b. Alternative — upload a zip (no Git remote)

Locally run `git archive -o /tmp/wwd_i.zip HEAD`, then run this instead of cell 2:

```python
from google.colab import files
files.upload()                          # choose wwd_i.zip
!rm -rf /content/wwd_i && mkdir -p /content/wwd_i && unzip -q wwd_i.zip -d /content/wwd_i
%cd /content/wwd_i
```


### 3. Provision Python 3.14 with `uv`


In [ ]:
!pip install -q uv
!uv venv --python 3.14
!.venv/bin/python --version

### 4. Install CUDA torch (override the CPU pin) + the package

`--no-config` ignores the repo's CPU torch pin; `--torch-backend=auto` detects the
T4's CUDA and selects matching wheels (fallback: `--index-url https://download.pytorch.org/whl/cu124`).
`datasets` is the MSWC streaming dep.


In [ ]:
!uv pip install --no-config --python .venv/bin/python torch torchaudio --torch-backend=auto
# our package (numpy/onnx/onnxruntime/soundfile/soxr) + the ONNX exporter + MSWC streaming (script-based -> datasets<4)
!uv pip install --python .venv/bin/python -e . onnxscript 'datasets<4'

### 5. Verify CUDA torch is active


In [ ]:
!.venv/bin/python -c "import torch; assert torch.cuda.is_available(), 'CUDA torch NOT active - is the runtime set to T4?'; print('OK:', torch.__version__, '|', torch.cuda.get_device_name(0))"

## Part A — MSWC scale-up (the real backbone)

Streams `MLCommons/ml_spoken_words` from the HF hub, materializes a capped subset
of 16 kHz clips, then trains. Streaming means we pull only the subset we keep, not
the full multi-GB corpus.


### A1. Inspect the dataset schema (do this first)

Prints one streamed example's fields. The default config is `en_wav` (English,
16 kHz; other languages follow `{lang}_wav` / `{lang}_opus` — `en_opus` is lighter
to stream but relies on opus decode). Confirm the word field is `keyword`; if not,
pass `--word-key <field>` to the prepare/train cells.

In [ ]:
!.venv/bin/python -m wwd_i.data.mswc --inspect

### A2. Materialize a subset

`--n-words` × `--clips-per-word` sets the vocabulary and clips/word. The stream is
shuffled so frequent words fill first; it stops once `--n-words` words are full and
prunes under-filled ones, so it streams somewhat more than it keeps. 500×80 keeps
~40k clips (~2–3 GB; allow ~15–30 min). Start smaller (`--n-words 300
--clips-per-word 60`) for a quicker first pass. Written under `/content/data/mswc`,
so it survives a training rerun.

In [ ]:
!.venv/bin/python -m wwd_i.data.mswc --root /content/data/mswc --n-words 500 --clips-per-word 80

### A3. Train the backbone

Bigger vocabulary → harder episodes (`--n-way 30`) and more steps. Watch the
`[probe @ N]` lines: backbone accuracy should beat the mel-baseline by a wide,
growing margin — and the absolute number should clear the bootstrap's **0.654**
(10-way) now that it has seen far more words.


In [ ]:
!.venv/bin/python -m wwd_i.train.train_backbone --dataset mswc --root /content/data/mswc --device cuda --mswc-held-out 50 --n-way 30 --probe-n-way 10 --k-shot 5 --q-query 5 --steps 8000 --eval-every 500 --probe-episodes 200 --embedding-dim 96 --out /content/artifacts/backbone_mswc.pt --onnx /content/artifacts/backbone_mswc.onnx

### A4. Download the MSWC artifacts


In [ ]:
from google.colab import files

files.download("/content/artifacts/backbone_mswc.onnx")
files.download("/content/artifacts/backbone_mswc.pt")

## Part B — Speech Commands bootstrap (optional reference)

The original ~24-word pipeline check. Already validated (held-out probe ~0.654 vs
mel-baseline ~0.22). First run downloads Speech Commands v2 (~2.3 GB).


In [ ]:
!.venv/bin/python -m wwd_i.train.train_backbone --device cuda --root /content/data/speech_commands --steps 4000 --eval-every 500 --probe-episodes 200 --n-way 12 --probe-n-way 10 --k-shot 5 --q-query 5 --lr 1e-3 --embedding-dim 96 --out /content/artifacts/backbone.pt --onnx /content/artifacts/backbone.onnx

## Interpreting the gate

- **Pass:** backbone probe accuracy clearly above the mel-baseline on held-out
  words → the embedding generalizes → drop `backbone_mswc.onnx` into the repo as
  the frozen backbone and move to Phase 3 (per-word data) / Phase 4 (head).
- **Weak / not better than the bootstrap:** widen vocabulary (`--n-words`),
  add clips (`--clips-per-word`), raise `--embedding-dim`, or train longer.
